# Use Python API to automate AutoAI deployment lifecycle

This notebook contains the steps and code to demonstrate support of AI Lifecycle features of the AutoAI model in watsonx.ai service. It contains steps and code to work with [ibm-watsonx-ai](https://pypi.python.org/pypi/ibm-watsonx-ai) SDK available in PyPI repository. It also introduces commands for training, persisting and deploying model, scoring it, updating the model and redeploying it.

Some familiarity with Python is helpful. This notebook uses Python 3.12.


## Learning goals

The learning goals of this notebook are:

-  List all deprecated and unsupported deployments.
-  Identify AutoAI models that need to be retrained.
-  Work with watsonx.ai experiments to re-train AutoAI models.
-  Persist an updated AutoAI model in watsonx.ai repository.
-  Redeploy model in-place.
-  Score sample records using client library.


## Contents

This notebook contains the following parts:

1. [Set up the environment](#1.-Set-up-the-environment)
2. [Deployments state check](#2.-Deployments-state-check)
3. [Identification of model requiring retraining](#3.-Identification-of-model-requiring-retraining)
4. [Experiment re-run](#4.-Experiment-re-run)
5. [Store the model in repository](#5.-Store-the-model-in-repository)
6. [Redeploy and score new version of the model](#6.-Redeploy-and-score-new-version-of-the-model)
7. [Cleanup](#7.-Cleanup)
8. [Summary and next steps](#8.-Summary-and-next-steps)

<a id="1.-Set-up-the-environment"></a>
## 1. Set up the environment

Before you use the sample code in this notebook, contact with your IBM Cloud Pak® for Data administrator and ask for your account credentials.

### Install dependencies
**Note:** `ibm-watsonx-ai` documentation can be found <a href="https://ibm.github.io/watsonx-ai-python-sdk/index.html" target="_blank" rel="noopener no referrer">here</a>.

In [1]:
%pip install -U wget | tail -n 1
%pip install "scikit-learn==1.6.1" | tail -n 1
%pip install -U autoai-libs | tail -n 1
%pip install -U ibm-watsonx-ai | tail -n 1

#### Define credentials

Authenticate the watsonx.ai Runtime service on IBM Cloud Pak® for Data. You need to provide the **admin's** `username` and the platform `url`.

In [2]:
username = "PASTE YOUR USERNAME HERE"
url = "PASTE THE PLATFORM URL HERE"

Use the **admin's** `api_key` to authenticate watsonx.ai Runtime services:

In [ ]:
import getpass

from ibm_watsonx_ai import Credentials

credentials = Credentials(
    username=username,
    api_key=getpass.getpass("Enter your watsonx.ai API key and hit enter: "),
    url=url,
    instance_id="openshift",
    version="5.3",
)

Alternatively you can use the **admin's** `password`:

In [3]:
import getpass

from ibm_watsonx_ai import Credentials

if "credentials" not in locals() or not credentials.api_key:
    credentials = Credentials(
        username=username,
        password=getpass.getpass("Enter your watsonx.ai password and hit enter: "),
        url=url,
        instance_id="openshift",
        version="5.3",
    )

#### Create `APIClient` instance

In [4]:
from ibm_watsonx_ai import APIClient

client = APIClient(credentials)

### Working with spaces

First of all, you need to create a space that will be used for your work. If you do not have space already created, you can use `{PLATFORM_URL}/ml-runtime/spaces?context=icp4data` to create one.

- Click New Deployment Space
- Create an empty space
- Go to space `Settings` tab
- Copy `space_id` and paste it below

**Tip**: You can also use SDK to prepare the space for your work. More information can be found [here](https://github.com/IBM/watsonx-ai-samples/blob/master/cpd5.3/notebooks/python_sdk/instance-management/Space%20management.ipynb).


You can use the `list` method to print all existing spaces.

In [ ]:
client.spaces.list(limit=10)

Extract all space IDs

In [5]:
space_ids = [
    space["metadata"]["id"] for space in client.spaces.get_details()["resources"]
]

space_ids[:5]

['9edd33a9-440a-41c3-bda0-55f082dfead6',
 '3b7108d1-9ee6-4cbd-a00a-fbbbc6725166',
 '4ae6199a-6f14-4151-beec-d8074092caee',
 'b0852cf1-438f-4258-a3c6-1d01e35a1c3a',
 'a3965abb-8227-4158-8428-8a0175523c87']

<a id="2.-Deployments-state-check"></a>
## 2. Deployments state check
Iterate over spaces and search for `deprecated` and `unsupported` deployments. Next, identify models requiring re-training.

In [6]:
from ibm_watsonx_ai.lifecycle import SpecStates

for space_id in space_ids[:5]:
    client.set.default_space(space_id)
    print(f"****** SPACE {space_id} ******")
    print(client.deployments.get_details(spec_state=SpecStates.DEPRECATED))
    print(client.deployments.get_details(spec_state=SpecStates.UNSUPPORTED))

****** SPACE 9edd33a9-440a-41c3-bda0-55f082dfead6 ******
{'resources': []}
{'resources': []}
****** SPACE 3b7108d1-9ee6-4cbd-a00a-fbbbc6725166 ******
{'resources': []}
{'resources': []}
****** SPACE 4ae6199a-6f14-4151-beec-d8074092caee ******
{'resources': []}
{'resources': []}
****** SPACE b0852cf1-438f-4258-a3c6-1d01e35a1c3a ******
{'resources': []}
{'resources': []}
****** SPACE a3965abb-8227-4158-8428-8a0175523c87 ******
{'resources': []}
{'resources': []}


You can also list deployments under particular space. The output contains `SPEC_STATE` and `SPEC_REPLACEMENT`. Set the working space.

In [7]:
deployment_space_id = "PASTE YOUR SPACE ID HERE"
client.set.default_space(deployment_space_id)

'SUCCESS'

List deployments under this space.

In [8]:
client.deployments.list()

,ID,NAME,STATE,CREATED,ARTIFACT_TYPE,SPEC_STATE,SPEC_REPLACEMENT
0,195dfc3a-98c8-44b0-affd-c239ac496c9a,Credit Risk Batch Deployment AutoAI,ready,2026-02-09T12:31:24.911Z,model,supported,
1,eafc6352-013f-4eba-b709-2925fa46dce2,Credit Risk Deployment AutoAI,ready,2026-02-09T12:28:44.470Z,model,supported,


<a id="3.-Identification-of-model-requiring-retraining"></a>
## 3. Identification of model requiring retraining
Pick up deployment of the AutoAI model you wish to retrain. 

**Hint**: You can also do that programatically in the loop sequence over spaces check (`Check the state of your deployments` cell).
**Hint**: You can also use software_specification information (model details) to identify models and deployments that are not yet deprecated but can be retrained (updated software specification is available).

In [9]:
deployment_id = "PASTE YOUR DEPLOYMENT ID HERE"

deployment_details = client.deployments.get_details(deployment_id)
deployed_model_id = deployment_details["entity"]["asset"]["id"]

deployed_model_id

Note: online_url is deprecated and will be removed in a future release. Use serving_urls instead.


'b31d2520-5a28-4a0f-bc6e-c8942bb43438'

#### Extract the deployed model's details (including the pipeline information).

In [10]:
import json

deployed_model_details = client.repository.get_model_details(deployed_model_id)
deployed_pipeline_id = deployed_model_details["entity"]["pipeline"]["id"]

deployed_pipeline_details = client.repository.get_details(deployed_pipeline_id)
experiment_params = deployed_pipeline_details["entity"]["document"]["pipelines"][0][
    "nodes"
][0]["parameters"]

optimization_params = experiment_params["optimization"]

print("Experiment parameters:", json.dumps(experiment_params, indent=2))
print("Optimization parameters:", json.dumps(optimization_params, indent=2))

Experiment parameters: {
  "drop_duplicates": true,
  "encoding": "utf-8",
  "input_file_separator": ",",
  "optimization": {
    "compute_pipeline_notebooks_flag": true,
    "label": "Risk",
    "learning_type": "binary",
    "retrain_on_holdout": true,
    "run_cognito_flag": true,
    "scorer_for_ranking": "roc_auc"
  },
  "output_logs": true,
  "stage_flag": true
}
Optimization parameters: {
  "compute_pipeline_notebooks_flag": true,
  "label": "Risk",
  "learning_type": "binary",
  "retrain_on_holdout": true,
  "run_cognito_flag": true,
  "scorer_for_ranking": "roc_auc"
}


#### Find the AutoAI experiment runs matching the extracted pipeline

Extract the project ID where the training took place.

**Tip:** For more information about using AutoAI with projects, see [this sample notebook](https://github.com/IBM/watsonx-ai-samples/blob/master/cpd5.3/notebooks/python_sdk/experiments/autoai/Use%20AutoAI%20with%20Watson%20Studio%20project.ipynb).

**Note:** If the training took place in a space, please update accordingly.

In [11]:
try:
    training_project_id = deployed_pipeline_details["metadata"]["tags"][0].split(".")[1]
except LookupError:
    training_project_id = input("Please enter your project_id (hit enter): ")

#### Extract AutoAI experiment `training_id`

The `training_id` is available in model's details.

In [12]:
run_id = deployed_model_details["entity"]["training_id"]
print("AutoAI experiment training_id found in model details:", run_id)

AutoAI experiment training_id found in model details: f07a3552-5c98-4c79-a24c-7cdd26e9922d


<a id="4.-Experiment-re-run"></a>
## 4. Experiment re-run

Set the training `project_id` (where data asset resides) to retrain AutoAI models.

In [13]:
from ibm_watsonx_ai.experiment import AutoAI

experiment = AutoAI(credentials, project_id=training_project_id)
optimizer = experiment.runs.get_optimizer(run_id=run_id)

In [14]:
from ibm_watsonx_ai.utils.autoai.errors import TestDataNotPresent

training_data_reference = optimizer.get_data_connections()
try:
    test_data_reference = optimizer.get_test_data_connections()
except TestDataNotPresent:
    test_data_reference = None

User defined (test / holdout) data is not present for this AutoAI experiment.
Reason: User specified test data was not present in this experiment. Try to use 'with_holdout_split' parameter for original training_data_references to retrieve test data.


In [15]:
train_details = optimizer.fit(
    training_data_references=training_data_reference,
    test_data_references=test_data_reference,
)

Training job a7fccb2d-eb39-4b1b-a5fe-ca3357f6287a completed: 100%|████████| [02:41<00:00,  1.62s/it]


### Explore experiment's results
Connect to finished experiment and preview the results.

In [16]:
optimizer.summary()

,Enhancements,Estimator,training_roc_auc_(optimized),holdout_average_precision,holdout_log_loss,training_accuracy,holdout_roc_auc,training_balanced_accuracy,training_f1,holdout_precision,training_average_precision,training_log_loss,holdout_recall,training_precision,holdout_accuracy,holdout_balanced_accuracy,training_recall,holdout_f1
Pipeline Name,,,,,,,,,,,,,,,,,,
Pipeline_4,"HPO, FE, HPO",XGBClassifier,0.853943,0.476558,0.415305,0.802590,0.829594,0.751099,0.859335,0.819407,0.917188,0.428142,0.915663,0.816230,0.809619,0.757233,0.907388,0.864865
Pipeline_5,"HPO, FE, HPO, Ensemble",BatchedTreeEnsembleClassifier(XGBClassifier),0.853943,0.476558,0.415305,0.802590,0.829594,0.751099,0.859335,0.819407,0.917188,0.428142,0.915663,0.816230,0.809619,0.757233,0.907388,0.864865
Pipeline_1,,XGBClassifier,0.848451,0.466398,0.356549,0.796120,0.829125,0.749861,0.852969,0.848066,0.912986,0.439419,0.924699,0.818963,0.839679,0.797679,0.890274,0.884726
Pipeline_2,HPO,XGBClassifier,0.848451,0.466398,0.356549,0.796120,0.829125,0.749861,0.852969,0.848066,0.912986,0.439419,0.924699,0.818963,0.839679,0.797679,0.890274,0.884726
Pipeline_3,"HPO, FE",XGBClassifier,0.848451,0.466398,0.356549,0.796120,0.829125,0.749861,0.852969,0.848066,0.912986,0.439419,0.924699,0.818963,0.839679,0.797679,0.890274,0.884726
Pipeline_6,,SnapBoostingMachineClassifier,0.850092,0.468067,0.403035,0.755522,0.825103,0.745869,0.808007,0.893333,0.915222,0.457473,0.807229,0.844414,0.807615,0.807806,0.775171,0.848101
Pipeline_7,HPO,SnapBoostingMachineClassifier,0.850092,0.468067,0.403035,0.755522,0.825103,0.745869,0.808007,0.893333,0.915222,0.457473,0.807229,0.844414,0.807615,0.807806,0.775171,0.848101
Pipeline_8,"HPO, FE",SnapBoostingMachineClassifier,0.845567,0.475130,0.441574,0.751732,0.818817,0.744172,0.804020,0.891892,0.912478,0.469330,0.795181,0.845317,0.799599,0.801782,0.767121,0.840764
Pipeline_9,"HPO, FE, HPO",SnapBoostingMachineClassifier,0.847878,0.484868,0.490099,0.742586,0.817356,0.745042,0.791775,0.861818,0.913129,0.472925,0.713855,0.855655,0.733467,0.743155,0.737588,0.780890


### Evaluate the best model locally

Load the model for test purposes.

**Hint:** The best model is returned automatically if no `pipeline_name` provided.

In [17]:
pipeline_name = "Pipeline_4"
pipeline_model = optimizer.get_pipeline(pipeline_name=pipeline_name, astype="sklearn")
pipeline_model

Pipeline(steps=[('featureunion',
                 FeatureUnion(transformer_list=[('float32_transform_140220915612128',
                                                 Pipeline(steps=[('numpycolumnselector',
                                                                  NumpyColumnSelector(columns=[0,
                                                                                               1,
                                                                                               2,
                                                                                               3,
                                                                                               5,
                                                                                               6,
                                                                                               7,
                                                                                               8,
                                                                                               9,
                                                                                               10,
                                                                                               11,
                                                                                               12,
                                                                                               13,
                                                                                               14,
                                                                                               15,
                                                                                               16,
                                                                                               17,
                                                                                               18,
                                                                                               19])),
                                                                 ('compressstrings',
                                                                  CompressStrings(compress_type='hash',
                                                                                  dtypes_list=['char_str',
                                                                                               'int_num',
                                                                                               'char_str',
                                                                                               'char_str',
                                                                                               'char_str',
                                                                                               'char_st...
                               feature_types=None, gamma=1, gpu_id=None,
                               grow_policy=None, importance_type='gain',
                               interaction_constraints=None,
                               learning_rate=0.031849646640891245, max_bin=None,
                               max_cat_threshold=None, max_cat_to_onehot=None,
                               max_delta_step=None, max_depth=5,
                               max_leaves=None, min_child_weight=19,
                               missing=nan, monotone_constraints=None,
                               multi_strategy=None, n_estimators=204, n_jobs=4,
                               nthread=None, ...))])

This cell constructs the cell scorer based on the experiment metadata.

In [18]:
from sklearn.metrics import get_scorer

scorer = get_scorer(optimization_params["scorer_for_ranking"])

#### Read the train and holdout data.

**Hint:** You can also use external test dataset.

In [19]:
connection = optimizer.get_data_connections()[0]
train_X, test_X, train_y, test_y = connection.read(with_holdout_split=True)

  Using cached pyarrow-23.0.0-cp312-cp312-macosx_12_0_x86_64.whl.metadata (3.0 kB)
Using cached pyarrow-23.0.0-cp312-cp312-macosx_12_0_x86_64.whl (35.8 MB)


#### Calculate the score

In [20]:
score = scorer(pipeline_model, test_X.values, test_y.values)
print(score)

0.8695079720077917


<a id="5.-Store-the-model-in-repository"></a>
## 5. Store the model in repository

Provide `pipeline_name` and `training_id`.

In [21]:
client.set.default_project(training_project_id)

Unsetting the space_id ...


'SUCCESS'

In [22]:
model_metadata = {
    client.repository.ModelMetaNames.NAME: "{0} - {1} - {2}".format(
        deployed_pipeline_details["metadata"]["name"],
        pipeline_name,
        pipeline_model.get_params()["steps"][-1][0],
    )
}
published_model = client.repository.store_model(
    model=pipeline_name,
    meta_props=model_metadata,
    training_id=train_details["metadata"]["id"],
)
updated_model_id = client.repository.get_model_id(published_model)
print("Re-trained model id", updated_model_id)

Re-trained model id 657df6af-20a5-4ea7-9769-4875e93a7848


List stored models.

In [23]:
client.repository.list_models()

,ID,NAME,CREATED,TYPE,SPEC_STATE,SPEC_REPLACEMENT
0,657df6af-20a5-4ea7-9769-4875e93a7848,Credit Risk Prediction - AutoAI - Pipeline_4 -...,2026-02-09T12:45:23Z,wml-hybrid_0.1,supported,
1,cb50d75b-644c-474e-9367-0601e89c6374,P2,2026-02-09T12:31:19Z,wml-hybrid_0.1,supported,
2,3c978320-b430-429c-84e7-f122d5e8f9ba,P1,2026-02-09T12:28:39Z,wml-hybrid_0.1,supported,


<a id="6.-Redeploy-and-score-new-version-of-the-model"></a>
## 6. Redeploy and score new version of the model

In this section, you'll learn how to redeploy new version of the model by using the watsonx.ai Client.

**Hint:** As a best practice please consider using the test space before moving to production.

```
promote(asset_id: str, source_project_id: str, target_space_id: str, rev_id: str = None)
```

### Promote model to deployment space

In [24]:
promoted_model_id = client.spaces.promote(
    asset_id=updated_model_id,
    source_project_id=training_project_id,
    target_space_id=deployment_space_id,
)

Check current deployment details before update.

In [25]:
client.set.default_space(deployment_space_id)
print(json.dumps(client.deployments.get_details(deployment_id), indent=2))

### Update the deployment with new model
**Note:** The update is asynchronous.

In [26]:
metadata = {
    client.deployments.ConfigurationMetaNames.ASSET: {
        "id": promoted_model_id,
    }
}

updated_deployment = client.deployments.update(deployment_id, changes=metadata)

Since ASSET is patched, deployment need to be restarted.


########################################################################

Deployment update for id: 'eafc6352-013f-4eba-b709-2925fa46dce2' started

########################################################################


updating....
ready


---------------------------------------------------------------------------------------------
Successfully finished deployment update, deployment_id='eafc6352-013f-4eba-b709-2925fa46dce2'
---------------------------------------------------------------------------------------------




Wait for the deployment update: 

In [27]:
import time

status = None
while status not in ("ready", "failed"):
    time.sleep(2)
    deployment_details = client.deployments.get_details(deployment_id)
    status = deployment_details["entity"]["status"].get("state")
    print(".", status, end=" ")

print("\nDeployment update finished with status: ", status)

Note: online_url is deprecated and will be removed in a future release. Use serving_urls instead.
. ready 
Deployment update finished with status:  ready


#### Get updated deployment details

In [28]:
print(json.dumps(client.deployments.get_details(deployment_id), indent=2))

### Score updated model
Create sample payload and score the deployed model.

In [29]:
scoring_payload = {"input_data": [{"values": test_X[:3]}]}

Use client.deployments.score() method to run scoring.

In [30]:
predictions = client.deployments.score(deployment_id, scoring_payload)

In [31]:
print(json.dumps(predictions, indent=2))

{
  "predictions": [
    {
      "fields": [
        "prediction",
        "probability"
      ],
      "values": [
        [
          "No Risk",
          [
            0.5054465532302856,
            0.49455347657203674
          ]
        ],
        [
          "No Risk",
          [
            0.9500458240509033,
            0.049954190850257874
          ]
        ],
        [
          "Risk",
          [
            0.473627507686615,
            0.526372492313385
          ]
        ]
      ]
    }
  ]
}


<a id="7.-Cleanup"></a>
## 7. Cleanup

If you want to clean up all created assets:
- experiments
- trainings
- pipelines
- models
- deployments

please follow up this sample [notebook](https://github.com/IBM/watsonx-ai-samples/blob/master/cpd5.3/notebooks/python_sdk/instance-management/Machine%20Learning%20artifacts%20management.ipynb).

<a id="8.-Summary-and-next-steps"></a>
## 8. Summary and next steps

You successfully completed this notebook! You learned how to use scikit-learn machine learning as well as watsonx.ai for model creation and deployment.

Check out our _<a href="https://ibm.github.io/watsonx-ai-python-sdk/samples.html" target="_blank" rel="noopener no referrer">Online Documentation</a>_ for more samples, tutorials, documentation, how-tos, and blog posts. 

### Authors

**Lukasz Cmielowski, PhD**, is a Senior Technical Staff Member at IBM with a track record of developing enterprise-level applications that substantially increases clients' ability to turn data into actionable knowledge.

**Dorota Lączak**, Python Software Developer in Watson Machine Learning AutoAI at IBM.

**Rafał Chrzanowski**, Software Engineer at watsonx.ai.

Copyright © 2023-2026 IBM. This notebook and its source code are released under the terms of the MIT License.